In [ ]:
import os
import pandas as pd
import requests

draft_ids = {"2024": "1125876441712902145","2023": "992213903041671169"}#, "2022": "863909738604314625"}
draft_dict = {}
name_to_pid_dict = {}

for year, id in draft_ids.items():
    url = "https://api.sleeper.app/v1/draft/" + id + "/picks"
    resp = requests.get(url=url)
    data = resp.json()
    prices_dict = {}
    for obj in data:
        meta = obj["metadata"]
        full_name = meta["first_name"] + " " + meta["last_name"]
        if full_name not in name_to_pid_dict:
            name_to_pid_dict[full_name] = meta["player_id"]
        prices_dict[full_name] = meta["amount"]
    draft_dict[year] = prices_dict



from collections import defaultdict as defaultdict

fantasy_pre = "Fantasy\YearlyRankings"
curr_year = 2025

old_rankings = os.listdir(fantasy_pre)

curr_ranking = ""

for rank in list(old_rankings):
    if str(curr_year) in rank:
        curr_ranking = rank
        old_rankings.remove(rank)
        continue
    elif "~" in rank or "2022" in rank or "Projected" in rank:
        old_rankings.remove(rank)

#print(curr_ranking)

#print(old_rankings)

price_dict = {}

inflation_dict = {}

price_dict["overall"] = defaultdict(list)

price_dict["pos"] = defaultdict(list)

price_dict["tier"] = defaultdict(list)

inflation_dict["overall"] = defaultdict(list)

inflation_dict["pos"] = defaultdict(list)

inflation_dict["tier"] = defaultdict(list)
print(old_rankings)
for name in old_rankings:
    year = ''.join([i for i in str(name) if i.isdigit()])
    print("Doing year " + year)

    num_seen = {
        "WR": 0,
        "TE": 0, 
        "RB": 0,
        "QB": 0
    }

    df = pd.read_excel(os.path.join(fantasy_pre, name), usecols = [0,4,5])
    for index, row in df.iterrows():
        split_name = str(row[0]).split()
        full_name = " ".join(val for val in split_name if val not in ["III", "II", "Jr."])
        pos_tier = str(row[1])
        pos_name = ''.join([i for i in str(pos_tier) if not i.isdigit()])
        if pos_name in ["K", "DEF", "D/ST"]:
            continue
        num_seen[pos_name] += 1
        pos_ranking = pos_name + str(num_seen[pos_name]).zfill(2)
        overall_ranking = str(sum(val for val in num_seen.values()))
        expect_value = float(row[2])
        if full_name not in name_to_pid_dict:
            continue

        if expect_value < 1:
            if expect_value <= 0:
                continue
            else:
                expect_value = 1

        try:
            price_drafted_for = float(draft_dict[year][full_name])
            price_dict["overall"][overall_ranking].append(price_drafted_for)
            price_dict["pos"][pos_ranking].append(price_drafted_for)
            price_dict["tier"][pos_tier].append(price_drafted_for)

            inflation = round((price_drafted_for/expect_value) * 100, 2)

            inflation_dict["overall"][overall_ranking].append(inflation)
            inflation_dict["pos"][pos_ranking].append(inflation)
            inflation_dict["tier"][pos_tier].append(inflation)
        except KeyError:
            print(full_name + " was not found, make sure this is expected")




df = pd.read_excel(os.path.join(fantasy_pre, curr_ranking), usecols = [0,4,5])

dic = {
"Name": "", "Positional Ranking": "", "Positional_Tier": "", "Expected_Price": ""
}

new_df = pd.DataFrame([dic])

num_seen = {
        "WR": 0,
        "TE": 0, 
        "RB": 0,
        "QB": 0
    }

for index, row in df.iterrows():
    split_name = str(row[0]).split()
    full_name = " ".join(val for val in split_name if val not in ["III", "II", "Jr."])
    pos_tier = str(row[1])
    pos_name = ''.join([i for i in str(pos_tier) if not i.isdigit()])
    if pos_name in ["K", "DEF", "D/ST"]:
        continue
    num_seen[pos_name] += 1
    pos_ranking = pos_name + str(num_seen[pos_name]).zfill(2)
    overall_ranking = str(sum(val for val in num_seen.values()))
    expect_value = float(row[2])

    inflation_vals = inflation_dict["tier"][pos_tier]
    
    if len(inflation_vals) == 0:
        inflated_value = expect_value
    else:
        inflated_value = round(expect_value * sum(inflation_vals)/len(inflation_vals)/100)

    new_df.loc[len(new_df)] = [full_name, pos_ranking, pos_tier, inflated_value]

new_df.to_excel(os.path.join(fantasy_pre, "ProjectedValues.xlsx"))
    



avgs = defaultdict(list)

for ranking, values in sorted(price_dict["pos"].items()):

    inflation_vals = inflation_dict["pos"][ranking]

    print("The " + ranking + " went for " + str(sum(values)/len(values)) + " on average, with " + str(sum(inflation_vals)/len(inflation_vals)) + "% change in price")






for ranking, values in sorted(price_dict["tier"].items()):

    inflation_vals = inflation_dict["tier"][ranking]

    print("The " + ranking + " tier went for " + str(sum(values)/len(values)) + " on average, with " + str(sum(inflation_vals)/len(inflation_vals)) + "% change in price")





